# Reproducing ParSeR on KoBLEX (EMNLP 2025)

This notebook re-runs the paper's proposed method, **ParSeR**, end to end on a subset of the KoBLEX benchmark and scores it with the repository's own metric functions.

**Pipeline (3 stages):**
1. **Parametric provisions**: the LLM writes what it thinks the relevant statute text says.
2. **Selection retrieval**: each generated provision is used as a BM25 query over 233,544 real Korean statute paragraphs (top 100), the candidates are reranked with a Korean cross-encoder, and the LLM picks 1 of the top 10.
3. **Answer generation**: the LLM answers using only the selected real provisions.

**What this notebook fixes compared with the repo's `parser/openai/` scripts:**
- `selection_retrieval.py` line 71 picks from `contexts_list[0]` (the first sub-query's candidates) for *every* sub-query. The original script (`parser/original/2.selection_retrieval_gpt.py`) correctly uses `re[i]`. We compute **both** from the same LLM outputs, so the comparison costs nothing extra.
- `run_pipeline.py` points to `source/koblex.jsonl`, but the data lives in `experiments/source/`.
- The API client uses a 10-second total timeout, and a failed call returns `None`, which crashes post-processing. We use 120 s and treat failures as empty answers.
- `utils.get_reranker` passes `default_activation_function=`, which current `sentence-transformers` rejects (it is now `activation_fn=`). We try the new name first.
- Prompts, parsing helpers, decoding settings (temperature 0.01, top_p 0.9, max_tokens 2000), BM25 settings, the reranker and the metrics are **imported or copied unchanged** from the repo.

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU*. The reranker needs the GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: switch the runtime to T4"
!git clone -q https://github.com/daehuikim/KoBLEX.git 2>/dev/null || echo "already cloned"
!cd KoBLEX && git checkout -q 1a2d225be33a70ebf91e2b6369acf27551dcfbb4 && git log -1 --format='Pinned commit: %h (%cd)'
!pip -q install bm25s PyStemmer sentence-transformers aiohttp tenacity deepeval nest_asyncio

## 1. Settings

- `BASE_URL` / `MODEL`: the paper uses OpenAI `gpt-4o`. Any OpenAI-compatible endpoint works, but **results from another model are not a reproduction of the GPT-4o numbers**. Say so in your report if you switch.
- `N_QUESTIONS`: start with 20 to see the real cost, then scale up to 226 if your budget allows.

In [ ]:
import os, getpass
BASE_URL = "https://api.openai.com/v1"      # OpenAI-compatible chat endpoint
MODEL = "gpt-4o"                            # paper setting
N_QUESTIONS = 20                            # 226 = full benchmark
SEED = 0
MAX_PARALLEL = 8
PRICE_IN, PRICE_OUT = 2.50, 10.00           # USD per 1M tokens; check your provider's current price
RUN_BIGRAM_VARIANT = True                   # extra experiment (section 6); re-runs only stage 2's LLM step

if not os.environ.get("LLM_API_KEY"):
    os.environ["LLM_API_KEY"] = getpass.getpass("API key: ")
ROOT = "/content/KoBLEX"
EXP = f"{ROOT}/experiments"
OUT = "/content/koblex_out"; os.makedirs(OUT, exist_ok=True)

## 2. Load data, pick a subset with the same 1/2/3-hop mix as the full set, and load the repo's own code

In [ ]:
import json, ast, re, sys, random, importlib.util
from collections import defaultdict

def load_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8")]
gold_all = load_jsonl(f"{EXP}/source/koblex.jsonl")
statutes = load_jsonl(f"{EXP}/source/statute.jsonl")
CTX = [s["hierarchy"] + s["content"] for s in statutes]     # same string the repo indexes

# stratified subset by hop count, deterministic
rng = random.Random(SEED)
by_hop = defaultdict(list)
for i, g in enumerate(gold_all): by_hop[str(g["n_hops"])].append(i)
picked = []
for h, idx in sorted(by_hop.items()):
    k = max(1, round(N_QUESTIONS * len(idx) / len(gold_all)))
    picked += sorted(rng.sample(idx, min(k, len(idx))))
picked = sorted(picked)[:N_QUESTIONS] if N_QUESTIONS < len(gold_all) else list(range(len(gold_all)))
data = [gold_all[i] for i in picked]
print(f"{len(data)} questions;", {h: sum(str(d['n_hops']) == h for d in data) for h in '123'})

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

P = f"{EXP}/parser/prompts"
pp_prompt  = load_module("pp_prompt",  f"{P}/parametric_provision.py")
sel_prompt = load_module("sel_prompt", f"{P}/selection_retrieval.py")
qa_prompt  = load_module("qa_prompt",  f"{P}/question_answering.py")
putils     = load_module("putils",     f"{EXP}/parser/openai/utils.py")      # parsing helpers
evalmod    = load_module("evalmod",    f"{EXP}/lf-eval/eval_pipeline.py")    # official metrics

## 3. LLM client

Same request body as `parser/openai/utils.py` (temperature 0.01, top_p 0.9, max_tokens 2000). Every response is cached to disk, so re-running a cell never pays twice.

In [ ]:
import asyncio, aiohttp, hashlib, nest_asyncio
from tenacity import retry, stop_after_attempt, wait_random_exponential
nest_asyncio.apply()
CACHE_PATH = f"{OUT}/llm_cache.jsonl"
CACHE = {}
if os.path.exists(CACHE_PATH):
    for l in open(CACHE_PATH, encoding="utf-8"):
        r = json.loads(l); CACHE[r["key"]] = r
USAGE = {"in": 0, "out": 0, "calls": 0, "failed": 0}

def _key(prompt): return hashlib.sha256((MODEL + "\x00" + prompt).encode()).hexdigest()

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(8), reraise=True)
async def _post(session, prompt):
    async with session.post(f"{BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['LLM_API_KEY']}", "Content-Type": "application/json"},
        json={"model": MODEL, "messages": [{"role": "user", "content": prompt}],
              "temperature": 0.01, "max_tokens": 2000, "top_p": 0.9}) as resp:
        js = await resp.json()
        if resp.status != 200: raise RuntimeError(f"{resp.status}: {str(js)[:300]}")
        return js

async def _one(session, sem, prompt):
    k = _key(prompt)
    if k in CACHE: return CACHE[k]["text"]
    async with sem:
        try:
            js = await _post(session, prompt)
        except Exception as e:
            USAGE["failed"] += 1; print("call failed:", str(e)[:200]); return ""
    text = js["choices"][0]["message"]["content"].strip()
    u = js.get("usage", {}); USAGE["in"] += u.get("prompt_tokens", 0); USAGE["out"] += u.get("completion_tokens", 0); USAGE["calls"] += 1
    CACHE[k] = {"key": k, "text": text}
    with open(CACHE_PATH, "a", encoding="utf-8") as f: f.write(json.dumps(CACHE[k], ensure_ascii=False) + "\n")
    return text

def call_llm_batch(prompts):
    async def run():
        sem = asyncio.Semaphore(MAX_PARALLEL)
        async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=120)) as s:
            return await asyncio.gather(*[_one(s, sem, p) for p in prompts])
    return asyncio.get_event_loop().run_until_complete(run())

def cost_so_far():
    c = USAGE["in"] / 1e6 * PRICE_IN + USAGE["out"] / 1e6 * PRICE_OUT
    return f"{USAGE['calls']} new calls, {USAGE['in']:,} in / {USAGE['out']:,} out tokens, ≈ ${c:.2f} (failed: {USAGE['failed']})"

# smoke test: one tiny call
print(call_llm_batch(["Reply with the single word: ok"]))
print(cost_so_far())

## 4. Stage 1: parametric provisions (prompt and parsing exactly as in `parser/openai/parametric_provision.py`)

In [ ]:
prompts1 = [f"{pp_prompt.SYSTEM_PROMPT}\n\n{pp_prompt.INSTRUCTION_PROMPT.substitute(background=d['background'], question=d['question'])}" for d in data]
raw1 = call_llm_batch(prompts1)
subs_all = []
for pred in raw1:
    block = putils.extract_first_list(pred)
    subs = putils.escape_quotes(putils.parse_list_block(block)) if block else []
    subs_all.append(subs if isinstance(subs, list) else [subs])
print("parametric provisions per question:", [len(s) for s in subs_all])
print("empty:", sum(not s for s in subs_all))
print("\nExample:", data[0]["question_eng"]); [print("  -", s[:150]) for s in subs_all[0]]
print("\n", cost_so_far(), f"→ projected for 226 Qs ≈ ${(USAGE['in']/1e6*PRICE_IN + USAGE['out']/1e6*PRICE_OUT) * 226 / len(data) * 3:.2f} (rough: stage 2 makes more calls)")

## 5. Stage 2: BM25 top-100 → cross-encoder rerank → LLM picks 1 of the top 10

BM25 is set up as in `parser/openai/utils.py` (`bm25s.tokenize(..., stopwords="en")`, no stemmer). The reranker is `dragonkue/bge-reranker-v2-m3-ko` with a sigmoid, scoring `(question, provision)` pairs as the repo does.

In [ ]:
import bm25s, torch, gc
from sentence_transformers import CrossEncoder

def build_index(tokenizer):
    r = bm25s.BM25()
    if tokenizer == "repo":
        r.index(bm25s.tokenize(CTX, stopwords="en", show_progress=False), show_progress=False)
        q = lambda s: bm25s.tokenize([s], show_progress=False)
    else:  # Korean character bigrams (our extra experiment)
        r.index([bigrams(c) for c in CTX], show_progress=False)
        q = lambda s: [bigrams(s)]
    return r, q

def bigrams(t):
    out = []
    for w in re.findall(r'[가-힣]+|[a-zA-Z0-9]+', t):
        out += [w] if len(w) < 2 else [w[i:i+2] for i in range(len(w) - 1)]
    return out

def load_reranker():
    # The repo passes default_activation_function=..., which newer sentence-transformers renamed to activation_fn
    name = "dragonkue/bge-reranker-v2-m3-ko"
    try:
        return CrossEncoder(name, activation_fn=torch.nn.Sigmoid(), device="cuda")
    except TypeError:
        return CrossEncoder(name, default_activation_function=torch.nn.Sigmoid(), device="cuda")

def stage2(tokenizer, reranker):
    retr, qtok = build_index(tokenizer)
    reranked = []
    for d, subs in zip(data, subs_all):
        per_q = []
        for sq in subs:
            idxs, _ = retr.retrieve(qtok(sq), k=100, show_progress=False)
            texts = [CTX[i] for i in idxs[0]]
            scores = reranker.predict([(d["question"], t) for t in texts], batch_size=50, show_progress_bar=False)
            per_q.append([t for _, t in sorted(zip(scores, texts), reverse=True)])
        reranked.append(per_q)
    prompts = []
    for d, per_q in zip(data, reranked):
        for cands in per_q:
            cand_str = "\n".join(f"{i}: {c}" for i, c in enumerate(cands[:10]))
            prompts.append(f"{sel_prompt.SYSTEM_PROMPT}\n\n{sel_prompt.INSTRUCTION_PROMPT.substitute(background=d['background'], question=d['question'], candidates=cand_str)}")
    outs = call_llm_batch(prompts)
    fixed, buggy, ptr = [], [], 0
    for per_q in reranked:
        f_q, b_q = [], []
        for i in range(len(per_q)):
            ans = outs[ptr].split("Answer:")[-1].strip(); ptr += 1
            choice = int(ans) if ans.isdigit() and 0 <= int(ans) <= 9 else 0     # repo's parsing rule
            f_q.append(per_q[i][choice])                                          # original script: re[i]
            b_q.append(per_q[0][min(choice, len(per_q[0]) - 1)])                  # refactored script: contexts_list[0]
        fixed.append(f_q); buggy.append(b_q)
    return fixed, buggy, reranked

reranker = load_reranker()
sel_fixed, sel_buggy, reranked_repo = stage2("repo", reranker)
print(cost_so_far())

## 6. Stage 3: answer only from the selected provisions (`parser/openai/question_answering.py`)

In [ ]:
def stage3(selections):
    prompts = [f"{qa_prompt.SYSTEM_PROMPT}\n\n{qa_prompt.INSTRUCTION_PROMPT.substitute(question=d['background'] + d['question'], context_str=chr(10).join(sel))}"
               for d, sel in zip(data, selections)]
    return [o.split("Answer:")[-1].strip() for o in call_llm_batch(prompts)]

ans_fixed = stage3(sel_fixed)
print(cost_so_far())

## 7. Score with the repo's metric functions, next to the authors' saved GPT-4o outputs for the same questions

In [ ]:
import pandas as pd
norm = lambda s: re.sub(r"\s+", " ", s).strip()
GOLD_CTX = []
for d in data:
    gc_ = ast.literal_eval(d["contexts"]) if isinstance(d["contexts"], str) else d["contexts"]
    GOLD_CTX.append([norm(c["hierarchy"] + c["content"]) for c in gc_])

def saved(name):
    rows = load_jsonl(f"{ROOT}/results/{name}.jsonl")
    out = []
    for i in picked:
        a = rows[i]["answers"]; a = a[0] if isinstance(a, list) else {"answer": a, "comb": []}
        out.append((str(a["answer"]), [norm(x) for x in a.get("comb") or []]))
    return out

def score(answers, sels, label):
    rows = []
    for hop in ["all", "1", "2", "3"]:
        ix = [i for i, d in enumerate(data) if hop == "all" or str(d["n_hops"]) == hop]
        if not ix: continue
        f1 = sum(evalmod.compute_token_f1(answers[i], data[i]["answer"])["f1"] for i in ix) / len(ix) if answers else float("nan")
        r = evalmod.evaluate_retrieval([[norm(x) for x in sels[i]] for i in ix], [GOLD_CTX[i] for i in ix])
        rows.append({"run": label, "hops": hop, "n": len(ix), "answer F1": round(f1 * 100, 1),
                     "ret P": round(r["precision"] * 100, 1), "ret R": round(r["recall"] * 100, 1),
                     "ret F1": round(r["f1"] * 100, 1), "ret EM": round(r["em"] * 100, 1)})
    return rows

sp = saved("parser_gpt4o"); so = saved("sp_or_gpt4o")
table = pd.DataFrame(
    score(ans_fixed, sel_fixed, f"ours: ParSeR ({MODEL}, bug fixed)") +
    score(None, sel_buggy, f"ours: ParSeR selection, refactored-code bug") +
    score([a for a, _ in sp], [c for _, c in sp], "authors' saved ParSeR GPT-4o") +
    score([a for a, _ in so], [c for _, c in so], "authors' saved standard retrieval GPT-4o"))
display(table[table.hops == "all"]); display(table)
table.to_csv(f"{OUT}/scores.csv", index=False)

## 8. Extra experiment: Korean-aware BM25 inside ParSeR

The repo tokenizes Korean with an English-oriented tokenizer, so words with particles attached (e.g. `손해배상청구권을`) only match exact surface forms. Outside ParSeR, switching BM25 to Korean character bigrams raised standard-retrieval F1 from 21.4 to 37.4 and recall@100 from 51.8 to 75.6 (no LLM involved). Here we test whether it also helps **inside** ParSeR. Stage 1 is reused from the cache, and only the selection calls are new.

In [ ]:
if RUN_BIGRAM_VARIANT:
    sel_bi, _, reranked_bi = stage2("bigram", reranker)
    ans_bi = stage3(sel_bi)
    t2 = pd.DataFrame(score(ans_bi, sel_bi, "ours: ParSeR + bigram BM25") + score(ans_fixed, sel_fixed, f"ours: ParSeR ({MODEL}, bug fixed)"))
    display(t2); t2.to_csv(f"{OUT}/scores_bigram.csv", index=False)
    print(cost_so_far())

## 9. Look at failures, then save everything

In [ ]:
for i, d in enumerate(data):
    hit = set(map(norm, sel_fixed[i])) & set(GOLD_CTX[i])
    if not hit:
        print(f"[{d['id']}] {d['question_eng']}")
        print("  gold:    ", [g[:60] for g in GOLD_CTX[i]])
        print("  selected:", [s[:60] for s in sel_fixed[i]])
        print("  generated provision:", [s[:80] for s in subs_all[i]], "\n")

with open(f"{OUT}/parser_ours.jsonl", "w", encoding="utf-8") as f:
    for d, subs, sel, a in zip(data, subs_all, sel_fixed, ans_fixed):
        f.write(json.dumps({"id": d["id"], "n_hops": d["n_hops"], "parametric_provisions": subs,
                            "selected_provisions": sel, "answer": a}, ensure_ascii=False) + "\n")
!cd /content && zip -qr koblex_out.zip koblex_out && echo "Saved /content/koblex_out.zip"
from google.colab import files; files.download("/content/koblex_out.zip")